# Projet final : les récoltes agricoles

**Équipe :** (nom de l'équipe et des 5 membres)

Une direction départementale de l'agriculture (fictive) vous confie trois ans de récoltes, de 2023 à 2025, par saison (printemps, automne), département et culture, avec la pluie tombée pendant chaque saison. Le ministère veut savoir quelles cultures pèsent le plus, où les rendements sont les meilleurs, et si la pluie explique les mauvaises années.

Ce notebook montre **votre démarche** : chaque étape dit ce que vous cherchez, le code, le résultat et ce que vous en concluez. Les graphiques doivent être visibles : avant de pousser, **Restart**, puis **Run All**.

Le tableau de bord (`app.py`) reprend ensuite vos résultats pour le décideur.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

brut = pd.read_csv("donnees/recoltes_brut.csv")

## 1. Diagnostic

Forme, types, valeurs manquantes, lignes en double, et toutes les écritures d'une même valeur dans les colonnes de texte. Notez chaque problème trouvé : vous le corrigerez à l'étape 2.

On regarde d'abord la taille du fichier et le type de chaque colonne.

In [2]:
print("Lignes, colonnes :", brut.shape)
brut.dtypes

Lignes, colonnes : (408, 6)


annee           int64
saison            str
departement       str
culture           str
surface_ha        str
production_t      str
dtype: object

`surface_ha` et `production_t` devraient être des nombres, or pandas les lit comme du texte (`str`) : certaines cellules contiennent donc autre chose que des chiffres. On cherche lesquelles.

In [3]:
for colonne in ["surface_ha", "production_t"]:
    valeurs = brut[colonne].dropna()
    pas_nombre = valeurs[pd.to_numeric(valeurs, errors="coerce").isna()]
    print(f"{colonne} : {len(pas_nombre)} valeurs qui ne sont pas des nombres, par exemple {list(pas_nombre.unique()[:6])}")

surface_ha : 22 valeurs qui ne sont pas des nombres, par exemple ['2 517', '541 ha', '1 127', '1 185', '1 748', '492 ha']
production_t : 50 valeurs qui ne sont pas des nombres, par exemple ['3753 t', '2299 t', '1306 t', '2394 t', '599 t', '767 t']


Valeurs manquantes et lignes en double :

In [4]:
print("Valeurs manquantes par colonne :")
print(brut.isna().sum())
print()
print("Lignes en double :", brut.duplicated().sum())

Valeurs manquantes par colonne :
annee            0
saison           0
departement      0
culture          0
surface_ha       0
production_t    15
dtype: int64

Lignes en double : 18


Toutes les écritures de chaque colonne de texte. Il devrait y avoir 2 saisons, 10 départements et 7 cultures.

In [5]:
for colonne in ["saison", "departement", "culture"]:
    ecritures = brut[colonne].unique()
    print(f"{colonne} : {len(ecritures)} écritures différentes")
    print(sorted(ecritures, key=str.lower))
    print()

saison : 2 écritures différentes
['Automne', 'Printemps']

departement : 44 écritures différentes
[' Artibonite', ' Centre', " Grand'Anse", ' Nord', ' Nord-Est', ' Nord-Ouest', ' Ouest', ' Sud', ' Sud-Est', 'Artibonite', 'ARTIBONITE', 'artibonite', 'Centre', 'centre', "Grand'Anse", "grand'anse", "GRAND'ANSE", "Grand'Anse ", 'Nippes', 'NIPPES', 'Nippes ', 'Nord', 'NORD', 'nord', 'Nord ', 'Nord-Est', 'nord-est', 'NORD-EST', 'Nord-Est ', 'NORD-OUEST', 'Nord-Ouest', 'nord-ouest', 'Nord-Ouest ', 'Ouest', 'ouest', 'OUEST', 'Ouest ', 'Sud', 'SUD', 'sud', 'Sud ', 'Sud-Est', 'sud-est', 'Sud-Est ']

culture : 33 écritures différentes
[' Banane', ' Haricot', ' Igname', ' Maïs', ' Pois congo', ' Riz', ' Sorgho', 'banane', 'Banane', 'BANANE', 'Banane ', 'Haricot', 'HARICOT', 'haricot', 'Haricot ', 'Igname', 'IGNAME', 'igname', 'Igname ', 'maïs', 'Maïs', 'MAÏS', 'Maïs ', 'Pois congo', 'POIS CONGO', 'pois congo', 'Pois congo ', 'Riz', 'Riz ', 'Sorgho', 'SORGHO', 'sorgho', 'Sorgho ']



In [6]:
print("Années :", sorted(brut["annee"].unique()))

Années : [np.int64(2023), np.int64(2024), np.int64(2025)]


**Problèmes trouvés**, à corriger à l'étape 2 :

1. **Nombres écrits comme du texte** : 22 surfaces et 50 productions contiennent une unité (`"541 ha"`, `"3753 t"`) ou un espace entre les milliers (`"2 517"`).
2. **Écritures multiples** : 44 écritures pour 10 départements et 33 pour 7 cultures, à cause d'espaces avant ou après (`" Maïs"`, `"Ouest "`) et de majuscules (`"NORD-OUEST"`, `"pois congo"`).
3. **Valeurs manquantes** : 15 productions sont vides.
4. **Doublons** : 18 lignes sont répétées à l'identique.

Les colonnes `annee` (3 valeurs) et `saison` (2 valeurs) sont déjà propres.

## 2. Nettoyage

Une copie `propre`, puis une cellule par problème, en comptant les lignes avant et après chaque suppression. Dites ce que vous supprimez et pourquoi.

**Contrôle :** à la fin, `propre.shape` vaut `(376, 6)`, sans valeur manquante ni doublon, avec 2 valeurs dans `saison`, 10 valeurs dans `departement`, 7 valeurs dans `culture`.

## 3. Jointures

Joignez `propre` avec `cultures.csv`, `pluies.csv`, et ajoutez les colonnes calculées dont vous aurez besoin. Comptez les lignes avant et après chaque `merge`.

**Contrôle :** 376 lignes après les jointures, autant qu'avant.

Enregistrez le tableau complet : c'est le fichier que lit le tableau de bord.

```python
complet.to_csv("donnees/recoltes_propre.csv", index=False)
```

## 4. Quelles cultures pèsent le plus, en tonnes puis en valeur (gourdes) ? Le classement change-t-il ?

**Ce qu'on en conclut :**

## 5. Où les rendements (tonnes par hectare) sont-ils les meilleurs, culture par culture ?

**Ce qu'on en conclut :**

## 6. La pluie explique-t-elle les rendements ? Pour quelles cultures ? Montrez-le avec des nuages de points.

**Ce qu'on en conclut :**

## 7. Quelles cultures progressent ou reculent entre 2023 et 2025 ?

**Ce qu'on en conclut :**

## 8. Rapportez la production à la population de chaque département : quels départements produisent le plus par habitant ?

**Ce qu'on en conclut :**

## 9. Conclusion

Cinq phrases au plus : ce qu'il faut retenir, ce que les données ne permettent pas de dire, ce qu'il faudrait mesurer ensuite.